In [1]:
import sys
sys.path.insert(0, '/mnt/Document/project/linkedin_scrapping/src')
from bs4 import BeautifulSoup
from models.html_identifier import HTMLIdentifier
from typing import Optional, Any
from scripts.extract.scrapping_source.get_list_preview_jobs import get_list_preview_jobs
from scripts.extract.scrapping_source.sample_html_data import sample_data
from constants.system_consts import GEMINI_JOB_URL_EXTRACTION_PROMPT
from selenium import webdriver;
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from scripts.extract.scrapping_source.scrapping_job_detail import scrapping_job_detail

/mnt/Document/project/linkedin_scrapping/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/mnt/Document/project/linkedin_scrapping/src/models/html_identifier.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [ ]:
import asyncio
import os
import json
import re
from playwright.async_api import async_playwright
role_mapping = {
        "Backend": ['backend', 'back-end', 'back end'],
        "Frontend": ['frontend', 'front-end', 'front end'],
        "Fullstack": ['fullstack', 'full-stack', 'full stack'],
        "Data Scientist": ['data scientist', 'data science'],
        "DevOps": ['devops', 'dev ops', 'dev-ops'],
        "Machine Learning Engineer": ['machine learning', 'ml engineer', 'ml engineering'],
        "Mobile Developer": ['mobile developer', 'mobile engineer', 'ios developer', 'android developer', 'mobile app developer', 'mobile engineering'],
        "QA Engineer": ['qa engineer', 'quality assurance', 'test engineer'],
        "AI Engineer": ['ai engineer', 'artificial intelligence'],
        "Data Analyst": ['data analyst', 'data analysis'],
        "Data Engineer": ['data engineer', 'data engineering', 'database engineer'],
        "Business Analyst": ['business analyst', 'business analysis'],
        "Embedded Engineer": ['embedded engineer', 'embedded systems', 'embedded software', 'embedded developer', 'embedded'],
    }
level_mapping = {
    'Internship': ['intern', 'internship'],
    'Fresher': ['fresher', 'new graduate', 'new grad', 'entry level'],
    'Junior': ['junior', 'jr', 'associate'],
    'Mid': ['mid level', 'mid-level', 'mid'],
    'Senior': ['senior', 'lead', 'principal', 'staff'],
}
# ===== Hàm EXTRACT DATA =====
# Định nghĩa trước khi sử dụng trong handle_response
import sys
sys.path.insert(0, '/mnt/Document/project/linkedin_scrapping/src')
from models.job import Job

def extract_job_detail_from_top_card(json_job_preview, json_job_detail, role_mapping, level_mapping):
    """
    Extract job details từ 2 response data sources và trả về Job object
    """
    # Parse JSON nếu là string
    if isinstance(json_job_preview, str):
        try:
            preview_data = json.loads(json_job_preview)
        except json.JSONDecodeError as e:
            print(f"JSON parsing error (preview): {e}")
            return None
    else:
        preview_data = json_job_preview
    
    if isinstance(json_job_detail, str):
        try:
            detail_data = json.loads(json_job_detail)
        except json.JSONDecodeError as e:
            print(f"JSON parsing error (detail): {e}")
            return None
    else:
        detail_data = json_job_detail
    
    job_info = {}
    
    # ===== PHẦN 1: Extract từ job_preview =====
    preview_included = preview_data.get("included", [])
    location_map = {}
    location_reference = None
    
    for item in preview_included:
        entity_urn = item.get("entityUrn", "")
        item_type = item.get("$type", "")
        
        if item_type == "com.linkedin.voyager.dash.common.Geo":
            location_map[entity_urn] = item.get("defaultLocalizedName", item.get("abbreviatedLocalizedName"))
        
        if item_type == "com.linkedin.voyager.dash.jobs.JobPosting":
            id_match = re.search(r'(\d{10})', entity_urn)
            if id_match:
                job_id = id_match.group(1)
                job_info["id"] = job_id
                job_info["link"] = f"https://www.linkedin.com/jobs/view/{job_id}"
                job_info["job_title"] = item.get("title")
                location_reference = item.get("*location")
        
        elif item_type == "com.linkedin.voyager.dash.jobs.JobPostingCard":
            if "primaryDescription" in item and item["primaryDescription"] is not None:
                job_info["company"] = item["primaryDescription"].get("text")
            
            if "tertiaryDescription" in item and item["tertiaryDescription"] is not None:
                tertiary_text = item["tertiaryDescription"].get("text", "")
                day_match = re.search(r'(Reposted|Posted)\s+([\w\s]+ago)', tertiary_text)
                if day_match:
                    job_info["day_posted"] = day_match.group(0)
                
                applicants_match = re.search(r'(\d+|Over \d+|Under \d+)\s+people\s+clicked\s+apply', tertiary_text)
                if applicants_match:
                    job_info["number_of_applicants"] = applicants_match.group(1)
    
    if location_reference and location_reference in location_map:
        job_info["location"] = location_map[location_reference]
    
    # ===== PHẦN 2: Extract Description từ job_detail =====
    detail_included = detail_data.get("included", [])
    
    for item in detail_included:
        item_type = item.get("$type", "")
        if item_type == "com.linkedin.voyager.dash.jobs.JobDescription":
            description_text = item.get("descriptionText")
            if description_text:
                if isinstance(description_text, dict):
                    job_info["description"] = description_text.get("text", "")
                else:
                    job_info["description"] = description_text
                break
    
    if not job_info:
        return None
    
    # ===== PHẦN 3: Mapping Role và Level dựa vào Job Title =====
    job_title = job_info.get("job_title", "").lower()
    mapped_role = None
    mapped_level = None
    
    # Mapping Role
    for role, keywords in role_mapping.items():
        for keyword in keywords:
            if keyword in job_title:
                mapped_role = role
                break
        if mapped_role:
            break
    
    # Mapping Level
    for level, keywords in level_mapping.items():
        for keyword in keywords:
            if keyword in job_title:
                mapped_level = level
                break
        if mapped_level:
            break
    
    # ===== PHẦN 4: Tạo và trả về Job object =====
    job_object = Job(
        identifyers=None,
        title=job_info.get("job_title"),
        company=job_info.get("company"),
        company_avatar_url=None,
        company_location=job_info.get("location"),
        location_working_type=None,
        working_type=None,
        date_posted=job_info.get("day_posted"),
        number_applicants=job_info.get("number_of_applicants"),
        job_url=job_info.get("link"),
        description=job_info.get("description"),
        salary=None,
        role=mapped_role,
        level=mapped_level
    )
    
    return job_object

# ===== Hàm FIND SCROLLABLE =====
async def find_scrollable_parent(page, child_selector):
    """Tìm element cha có thể scroll"""
    scroll_container = await page.evaluate(f"""
        () => {{
            let element = document.querySelector('{child_selector}');
            if (!element) return null;
            
            let current = element;
            while (current) {{
                const scrollHeight = current.scrollHeight;
                const clientHeight = current.clientHeight;
                if (scrollHeight > clientHeight) {{
                    return current;
                }}
                current = current.parentElement;
            }}
            return document.body;
        }}
    """)
    return scroll_container

# ===== Hàm MAIN SCRAPING =====

async def run():
    async with async_playwright() as p:
        user_data_dir = os.path.join(os.getcwd(), "linkedin_profile")
        print("--- Đang khởi động trình duyệt ---")
        
        context = await p.chromium.launch_persistent_context(
            user_data_dir,
            headless=False,
            args=[
                "--start-maximized",
                "--disable-blink-features=AutomationControlled"
            ],
            no_viewport=True
        )

        page = context.pages[0]
        
        # ===== VARIABLES LƯU TRỮ DỮ LIỆU =====
        collected_jobs = []
        current_job_preview = None
        current_job_detail = None

        async def handle_response(response):
            nonlocal current_job_preview, current_job_detail
            
            if "graphql" in response.url and "TOP_CARD" in response.url:
                try:
                    data = await response.json()
                    current_job_preview = data
                    # ===== GỌI EXTRACT KHI CÓ ĐỦ 2 GÓI TIN =====
                    if current_job_preview is not None and current_job_detail is not None:
                        print("\n[✨] Đã nhận được cả 2 gói tin, đang extract dữ liệu...")
                        job_object = extract_job_detail_from_top_card(current_job_preview, current_job_detail, role_mapping, level_mapping)
                        if job_object:
                            collected_jobs.append(job_object)
                            job_id = job_object.job_url.split('/')[-1] if job_object.job_url else "N/A"
                            print(f"✅ Đã thêm job vào list: [{job_id}] {job_object.title}")
                        current_job_preview = None
                        current_job_detail = None
                except Exception as e:
                    print(f"❌ Lỗi: {e}")
            elif "graphql" in response.url and "JOB_DESCRIPTION_CARD" in response.url:
                try:
                    data = await response.json()
                    current_job_detail = data
                    # ===== GỌI EXTRACT KHI CÓ ĐỦ 2 GÓI TIN =====
                    if current_job_preview is not None and current_job_detail is not None:
                        print("\n[✨] Đã nhận được cả 2 gói tin, đang extract dữ liệu...")
                        job_object = extract_job_detail_from_top_card(current_job_preview, current_job_detail, role_mapping, level_mapping)
                        if job_object:
                            collected_jobs.append(job_object)
                            job_id = job_object.job_url.split('/')[-1] if job_object.job_url else "N/A"
                            print(f"✅ Đã thêm job vào list: [{job_id}] {job_object.title}")
                        current_job_preview = None
                        current_job_detail = None
                        
                except Exception as e:
                    print(f"❌ Lỗi: {e}")
            
        page.on("response", handle_response)

        # 1. Truy cập trang Jobs
        await page.goto("https://www.linkedin.com/jobs/", wait_until="domcontentloaded")

        # 2. Kiểm tra đăng nhập
        if await page.get_by_role("link", name="Sign in").is_visible():
            print("⚠️ BẠN CHƯA ĐĂNG NHẬP!")
            print("Vui lòng đăng nhập thủ công...")
            await page.wait_for_url("**/linkedin.com/feed/**", timeout=0)
            print("✅ Đăng nhập thành công!")
            await page.goto("https://www.linkedin.com/jobs/search/")
        else:
            print("✅ Đã nhận diện phiên đăng nhập cũ.")
            await page.goto("https://www.linkedin.com/jobs/search/")

        # 3. Chờ danh sách Jobs
        print("Đang tải danh sách công việc...")
        try:
            await page.wait_for_selector(".job-card-container", timeout=10000)
        except:
            print("❌ Không tìm thấy danh sách Jobs")
            return

        # 4. Tìm element scrollable
        print("Đang tìm container scrollable...")
        scroll_container = await find_scrollable_parent(page, ".job-card-container")
        if not scroll_container:
            print("❌ Không tìm thấy element scrollable")
            return
        print(f"✅ Tìm thấy element scrollable")
        
        # 5. Scroll để load thêm jobs
        for _ in range(3):
            await page.evaluate("""
                () => {
                    let element = document.querySelector('.job-card-container');
                    if (!element) return;
                    let current = element;
                    while (current) {
                        const scrollHeight = current.scrollHeight;
                        const clientHeight = current.clientHeight;
                        if (scrollHeight > clientHeight) {
                            current.scrollTop += 1000;
                            break;
                        }
                        current = current.parentElement;
                    }
                }
            """)
            await page.wait_for_timeout(1000)

        # 6. Duyệt từng job
        job_cards_selector = ".job-card-container"
        job_items = page.locator(job_cards_selector)
        count = await job_items.count()
        
        print(f"\n🚀 Tìm thấy {count} công việc. Bắt đầu duyệt...")

        for i in range(count):
            try:
                current_job_preview = None
                current_job_detail = None
                
                job_card = job_items.nth(i)
                await job_card.scroll_into_view_if_needed()
                await job_card.click()

                # Chờ để nhận cả 2 gói tin (timeout 5 giây)
                for wait_count in range(50):
                    if current_job_preview is not None and current_job_detail is not None:
                        break
                    await page.wait_for_timeout(100)
                
                if current_job_preview is None or current_job_detail is None:
                    print(f"⚠️ Job thứ {i+1}: Không nhận được đủ 2 gói tin")
                
                await page.wait_for_timeout(2000)
                
            except Exception as e:
                print(f"⚠️ Lỗi ở Job thứ {i+1}: {e}")
                continue

        # 7. Hiển thị kết quả
        print("\n" + "=" * 70)
        print(f"✨ Hoàn thành duyệt danh sách. Tổng cộng {len(collected_jobs)} job đã collect")
        print("=" * 70)
        
        if collected_jobs:
            print("\n📋 DANH SÁCH CÁC JOB ĐÃ COLLECT:")
            for idx, job in enumerate(collected_jobs, 1):
                print(f"\n[{idx}] {job.title}")
                print(f"    Công ty: {job.company}")
                print(f"    Địa điểm: {job.company_location}")
                print(f"    Ngày đăng: {job.date_posted}")
                print(f"    Ứng viên: {job.number_applicants}")
                print(f"    Role: {job.role}")
                print(f"    Level: {job.level}")
                print(f"    Link: {job.job_url}")
                if job.description:
                    print(f"    Mô tả: {job.description[:100]}...")
        else:
            print("⚠️ Không có job nào được collect")
        
        input("\nNhấn Enter để đóng trình duyệt...")
        await context.close()

await run()


/mnt/Document/project/linkedin_scrapping/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/mnt/Document/project/linkedin_scrapping/src/models/html_identifier.py:5: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


--- Đang khởi động trình duyệt ---
✅ Đã nhận diện phiên đăng nhập cũ.
Đang tải danh sách công việc...
Đang tìm container scrollable...

[✨] Đã nhận được cả 2 gói tin, đang extract dữ liệu...
✅ Đã thêm job vào list: [4348234933] Android Developer
✅ Tìm thấy element scrollable

🚀 Tìm thấy 25 công việc. Bắt đầu duyệt...


In [ ]:
# Extract top card job detail

import sys
sys.path.insert(0, '/mnt/Document/project/linkedin_scrapping/src')
from models.job import Job

def extract_job_detail_from_top_card(json_job_preview, json_job_detail, role_mapping, level_mapping):
    """
    Extract job details từ 2 response data sources và trả về Job object
    
    Parameters:
        json_job_preview: JSON string hoặc dict từ TOP_CARD/JOB_PREVIEW response (last_job_preview.json)
        json_job_detail: JSON string hoặc dict từ JOB_DESCRIPTION_CARD response (last_job_detail.json)
        role_mapping: Dict mapping job roles (ví dụ: {'Backend': ['backend', 'back-end'], ...})
        level_mapping: Dict mapping seniority levels (ví dụ: {'Junior': ['junior', 'jr'], ...})
    
    Returns:
        Job: Object chứa các thông tin công việc đã extract
    """
    import re
    import json
    
    # Parse JSON nếu là string
    if isinstance(json_job_preview, str):
        try:
            preview_data = json.loads(json_job_preview)
        except json.JSONDecodeError as e:
            print(f"JSON parsing error (preview): {e}")
            return {}
    else:
        preview_data = json_job_preview
    
    if isinstance(json_job_detail, str):
        try:
            detail_data = json.loads(json_job_detail)
        except json.JSONDecodeError as e:
            print(f"JSON parsing error (detail): {e}")
            return {}
    else:
        detail_data = json_job_detail
    
    job_info = {}
    
    # ===== PHẦN 1: Extract từ job_preview =====
    preview_included = preview_data.get("included", [])
    
    # Build location map từ Geo items
    location_map = {}
    location_reference = None
    
    for item in preview_included:
        entity_urn = item.get("entityUrn", "")
        item_type = item.get("$type", "")
        
        # Lưu location từ Geo
        if item_type == "com.linkedin.voyager.dash.common.Geo":
            location_map[entity_urn] = item.get("defaultLocalizedName", item.get("abbreviatedLocalizedName"))
        
        # 1. Lấy ID và Title từ JobPosting
        if item_type == "com.linkedin.voyager.dash.jobs.JobPosting":
            id_match = re.search(r'(\d{10})', entity_urn)
            if id_match:
                job_id = id_match.group(1)
                job_info["id"] = job_id
                job_info["link"] = f"https://www.linkedin.com/jobs/view/{job_id}"
                job_info["job_title"] = item.get("title")
                # Lưu location reference
                location_reference = item.get("*location")
        
        # 2. Lấy Company, số ứng viên, ngày đăng từ JobPostingCard
        elif item_type == "com.linkedin.voyager.dash.jobs.JobPostingCard":
            # Công ty
            if "primaryDescription" in item and item["primaryDescription"] is not None:
                job_info["company"] = item["primaryDescription"].get("text")
            
            # Thời gian đăng và số ứng viên từ tertiaryDescription
            if "tertiaryDescription" in item and item["tertiaryDescription"] is not None:
                tertiary_text = item["tertiaryDescription"].get("text", "")
                
                # Trích xuất phần ngày đăng
                day_match = re.search(r'(Reposted|Posted)\s+([\w\s]+ago)', tertiary_text)
                if day_match:
                    job_info["day_posted"] = day_match.group(0)
                else:
                    job_info["day_posted"] = None
                
                # Trích xuất số ứng viên
                applicants_match = re.search(r'(\d+|Over \d+|Under \d+)\s+people\s+clicked\s+apply', tertiary_text)
                if applicants_match:
                    job_info["number_of_applicants"] = applicants_match.group(1)
                else:
                    job_info["number_of_applicants"] = None
    
    # Resolve location từ reference
    if location_reference and location_reference in location_map:
        job_info["location"] = location_map[location_reference]
    
    # ===== PHẦN 2: Extract Description từ job_detail =====
    detail_included = detail_data.get("included", [])
    
    for item in detail_included:
        item_type = item.get("$type", "")
        
        # Tìm JobDescription để lấy description
        if item_type == "com.linkedin.voyager.dash.jobs.JobDescription":
            # Mô tả công việc nằm ở descriptionText - có thể là dict hoặc string
            description_text = item.get("descriptionText")
            if description_text:
                # Nếu là dict, lấy field "text"
                if isinstance(description_text, dict):
                    job_info["description"] = description_text.get("text", "")
                else:
                    # Nếu là string, sử dụng trực tiếp
                    job_info["description"] = description_text
                break
    
    # Đảm bảo các field quan trọng tồn tại
    if not job_info:
        print("❌ Không tìm thấy dữ liệu job trong response")
        return None
    
    # ===== PHẦN 3: Mapping Role và Level dựa vào Job Title =====
    job_title = job_info.get("job_title", "").lower()
    mapped_role = None
    mapped_level = None
    
    # Mapping Role
    for role, keywords in role_mapping.items():
        for keyword in keywords:
            if keyword in job_title:
                mapped_role = role
                break
        if mapped_role:
            break
    
    # Mapping Level
    for level, keywords in level_mapping.items():
        for keyword in keywords:
            if keyword in job_title:
                mapped_level = level
                break
        if mapped_level:
            break
    
    # ===== PHẦN 4: Tạo và trả về Job object =====
    job_object = Job(
        identifyers=None,
        title=job_info.get("job_title"),
        company=job_info.get("company"),
        company_avatar_url=None,
        company_location=job_info.get("location"),
        location_working_type=None,
        working_type=None,
        date_posted=job_info.get("day_posted"),
        number_applicants=job_info.get("number_of_applicants"),
        job_url=job_info.get("link"),
        description=job_info.get("description"),
        salary=None,
        role=mapped_role,
        level=mapped_level
    )
    
    return job_object

In [23]:
# Test hàm với dữ liệu từ last_job_preview.json và last_job_detail.json
import os

preview_data = None
detail_data = None

# Đọc dữ liệu preview
if os.path.exists("last_job_preview.json"):
    with open("last_job_preview.json", "r", encoding="utf-8") as f:
        preview_data = json.load(f)
    print("✅ Đã load last_job_preview.json")
else:
    print("⚠️ File last_job_preview.json chưa tồn tại")

# Đọc dữ liệu detail
if os.path.exists("last_job_detail.json"):
    with open("last_job_detail.json", "r", encoding="utf-8") as f:
        detail_data = json.load(f)
    print("✅ Đã load last_job_detail.json")
else:
    print("⚠️ File last_job_detail.json chưa tồn tại")

# Test hàm
if preview_data and detail_data:
    # Gọi hàm với role_mapping và level_mapping từ cell 2
    result = extract_job_detail_from_top_card(preview_data, detail_data, role_mapping, level_mapping)
    
    if result:
        print("\n📋 Thông tin công việc đã extract:")
        print(f"{'─' * 60}")
        print(f"  Title: {result.title}")
        print(f"  Company: {result.company}")
        print(f"  Location: {result.company_location}")
        print(f"  Date Posted: {result.date_posted}")
        print(f"  Number Applicants: {result.number_applicants}")
        print(f"  Job URL: {result.job_url}")
        print(f"  Role: {result.role}")
        print(f"  Level: {result.level}")
        if result.description:
            print(f"  Description: {result.description[:100]}...")
        print(f"{'─' * 60}")
    else:
        print("❌ Không extract được dữ liệu job")
else:
    print("❌ Không đủ dữ liệu để test. Cần cả preview và detail JSON files")



✅ Đã load last_job_preview.json
✅ Đã load last_job_detail.json


NameError: name 'role_mapping' is not defined

In [4]:
import json
import sys
import re
sys.path.insert(0, '/mnt/Document/project/linkedin_scrapping/src')
from scripts.extract.scrapping_source.raw_data import raw_data

def extract_linkedin_jobs(json_data):
    if isinstance(json_data, str):
        try:
            data = json.loads(json_data)
        except json.JSONDecodeError as e:
            print(f"JSON parsing error: {e}")
            print("Attempting to clean control characters...")
            try:
                import re
                cleaned = json_data
                cleaned = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f]', '', cleaned)
                data = json.loads(cleaned)
                print("Successfully parsed after cleaning")
            except json.JSONDecodeError as e2:
                print(f"Still failed to parse: {e2}")
                return []
    else:
        data = json_data

    # Kho lưu trữ tạm thời để gộp dữ liệu theo ID
    # Cấu trúc: { "4353634833": { "title": "...", "company": "...", ... } }
    job_map = {}

    included = data.get("included", [])

    # Duyệt qua mảng included để thu thập các mảnh dữ liệu
    for item in included:
        entity_urn = item.get("entityUrn", "")
        
        # Tách lấy ID số từ chuỗi URN (ví dụ: 4353634833)
        # URN có thể là: urn:li:fsd_jobPosting:4353634833 
        # hoặc: urn:li:fsd_jobPostingCard:(4353634833,...)
        id_match = re.search(r'(\d{10})', entity_urn)
        if not id_match:
            continue
        
        job_id = id_match.group(1)
        
        if job_id not in job_map:
            job_map[job_id] = {"id": job_id, "link": f"https://www.linkedin.com/jobs/view/{job_id}"}

        # 1. Lấy Title từ loại JobPosting
        if item.get("$type") == "com.linkedin.voyager.dash.jobs.JobPosting":
            job_map[job_id]["title"] = item.get("title")
            job_map[job_id]["is_reposted"] = item.get("repostedJob")

        # 2. Lấy Company và Location từ loại JobPostingCard
        elif item.get("$type") == "com.linkedin.voyager.dash.jobs.JobPostingCard":
            # Tên công ty thường nằm ở primaryDescription
            if "primaryDescription" in item:
                job_map[job_id]["company"] = item["primaryDescription"].get("text")
            
            # Địa điểm thường nằm ở secondaryDescription
            if "secondaryDescription" in item:
                job_map[job_id]["location"] = item["secondaryDescription"].get("text")
            
            # Thời gian đăng
            if "formattedTimestamp" in item:
                job_map[job_id]["posted_at"] = item["formattedTimestamp"].get("text")

    # Trả về danh sách các object đã hoàn thiện, bỏ qua các object thiếu title (nếu có)
    return [job for job in job_map.values() if "title" in job]

# --- VÍ DỤ SỬ DỤNG ---
# Giả sử 'raw_data' là biến chứa toàn bộ nội dung JSON bạn đã copy
final_jobs = extract_linkedin_jobs(raw_data)

for job in final_jobs:
    print(f"ID: {job['id']}")
    print(f"Chức danh: {job.get('title')}")
    print(f"Công ty: {job.get('company', 'N/A')}")
    print(f"Địa điểm: {job.get('location', 'N/A')}")
    print(f"Link: {job['link']}")
    print("-" * 30)

JSON parsing error: Invalid control character at: line 76 column 28 (char 3425)
Attempting to clean control characters...
Successfully parsed after cleaning
ID: 4353634833
Chức danh: Mobile Engineer
Công ty: MoMo (M_Service)
Địa điểm: Ho Chi Minh City, Vietnam (On-site)
Link: https://www.linkedin.com/jobs/view/4353634833
------------------------------
ID: 4339408766
Chức danh: Frontend Software Engineer II
Công ty: Axon
Địa điểm: Ho Chi Minh City, Ho Chi Minh City, Vietnam
Link: https://www.linkedin.com/jobs/view/4339408766
------------------------------
ID: 4339791071
Chức danh: Design Verification, Senior Staff Engineer
Công ty: Marvell Technology
Địa điểm: Ho Chi Minh City, Ho Chi Minh City, Vietnam
Link: https://www.linkedin.com/jobs/view/4339791071
------------------------------
ID: 4200551721
Chức danh: Quality Assurance Automation Engineer
Công ty: Astek Vietnam
Địa điểm: Ho Chi Minh City, Vietnam (On-site)
Link: https://www.linkedin.com/jobs/view/4200551721
--------------------